# Chapter 1 Walkthrough — Probability and Inference

This notebook walks through the Chapter 1 Bayesian update loop applied to revenue growth estimation.

The goal is to go from the three abstract steps Gelman defines in Section 1.1 to a running PyMC model that produces a full posterior distribution over a company's true revenue growth rate.

**Sections covered:** 1.1, 1.3, 1.6, 1.9, 1.10

---

## Setup

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Makes plots render inline in the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['font.family'] = 'sans-serif'

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

---

## Step 1 — The Data

These are 8 quarters of observed year-over-year revenue growth rates for a hypothetical company.

In the Bayesian framework, this is $y$ — the observed data that we'll use to update our prior belief about the true underlying growth rate $\theta$.

In [ ]:
quarters = ['Q1 2022', 'Q2 2022', 'Q3 2022', 'Q4 2022',
            'Q1 2023', 'Q2 2023', 'Q3 2023', 'Q4 2023']

# y — observed quarterly revenue growth rates
y = np.array([0.082, 0.074, 0.091, 0.068, 0.055, 0.101, 0.079, 0.072])

print("Observed quarterly growth rates:")
for q, g in zip(quarters, y):
    print(f"  {q}: {g:.1%}")
print(f"\nSample mean:  {y.mean():.2%}")
print(f"Sample std:   {y.std():.2%}")

# Quick plot
fig, ax = plt.subplots()
ax.bar(quarters, y, color='#1F3A7A', alpha=0.8)
ax.axhline(y.mean(), color='#C8960C', linewidth=2, linestyle='--', label=f'Mean: {y.mean():.1%}')
ax.set_title('Observed Quarterly Revenue Growth (y)', fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.tick_params(axis='x', rotation=35)
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 2 — The Prior (Section 1.6)

Before looking at the data, we need to encode what we believe about $\theta$ — the true underlying growth rate.

We don't make this up randomly. We use **structural knowledge** — specifically, the distribution of revenue growth rates across comparable public companies in this sector. This is the Section 1.6 idea: use industry-level patterns as a structural prior before company-specific data is sufficient to speak for itself.

We're saying:

$$\theta \sim \text{Normal}(\mu = 0.07, \sigma = 0.03)$$

In plain English: we believe the true growth rate is probably around 7%, and we're willing to be wrong by about 3% in either direction before the data comes in.

In [ ]:
# Prior hyperparameters — sector-level structural knowledge
prior_mu = 0.07     # sector median growth rate
prior_sigma = 0.03  # sector spread

# Draw samples from the prior to visualize it
prior_samples = np.random.normal(prior_mu, prior_sigma, size=10000)

fig, ax = plt.subplots()
ax.hist(prior_samples, bins=80, color='#6B8CCC', alpha=0.8, edgecolor='white', linewidth=0.2)
ax.axvline(prior_mu, color='#C8960C', linewidth=2, label=f'Prior mean: {prior_mu:.0%}')
ax.set_title('Prior Distribution over True Growth Rate $\\theta$\n(Before seeing any company data)', fontweight='bold')
ax.set_xlabel('Revenue Growth Rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.show()

print(f"Prior: theta ~ Normal({prior_mu:.0%}, {prior_sigma:.0%})")
print(f"Prior 90% interval: [{np.percentile(prior_samples, 5):.1%}, {np.percentile(prior_samples, 95):.1%}]")

---

## Step 3 — The Model (Sections 1.3, 1.9)

Now we specify the full Bayesian model in PyMC. Three components:

1. **Prior** on $\theta$ — the true growth rate
2. **Prior** on $\sigma$ — the quarter-to-quarter noise level  
3. **Likelihood** — how probable is $y$ given $\theta$

PyMC builds the computation graph. NUTS handles the sampling. We never need to compute the normalizing constant $p(y)$ — the sampler bypasses it entirely.

In [ ]:
with pm.Model() as growth_model:

    # --- PRIOR ---
    # theta: true underlying revenue growth rate
    # Centered on sector median, uncertainty encoded as sigma
    mu_growth = pm.Normal('mu_growth', mu=prior_mu, sigma=prior_sigma)

    # Quarter-to-quarter noise — must be positive, so HalfNormal
    sigma_growth = pm.HalfNormal('sigma_growth', sigma=0.02)

    # --- LIKELIHOOD ---
    # p(y | theta) — how probable is each observed quarter given the true growth rate
    observed = pm.Normal('observed', mu=mu_growth, sigma=sigma_growth, observed=y)

    # --- POSTERIOR ---
    # Sample using NUTS. This IS the posterior — no closed form needed.
    trace = pm.sample(draws=2000, tune=1000, target_accept=0.9,
                      return_inferencedata=True, progressbar=True)

    # Posterior predictive — forward scenarios for next quarter
    ppc = pm.sample_posterior_predictive(trace)

---

## Step 4 — Diagnostics (Section 1.9)

Before looking at results, always check convergence. $\hat{R}$ (R-hat) should be below 1.01 for all parameters. If it's not, the chains haven't converged and the posterior samples can't be trusted.

In [ ]:
print("=== Convergence Diagnostics ===")
summary = az.summary(trace, var_names=['mu_growth', 'sigma_growth'])
print(summary)

rhat = az.rhat(trace)
r_mu = float(rhat['mu_growth'].values)
r_sigma = float(rhat['sigma_growth'].values)

print(f"\nR-hat mu_growth:    {r_mu:.4f}  {'✅ Converged' if r_mu < 1.01 else '❌ Not converged'}")
print(f"R-hat sigma_growth: {r_sigma:.4f}  {'✅ Converged' if r_sigma < 1.01 else '❌ Not converged'}")

---

## Step 5 — The Posterior (Sections 1.1, 1.10)

This is the output. The posterior distribution over $\theta$ — the true underlying growth rate — updated by 8 quarters of company-specific data on top of the sector prior.

This is what you serve to clients. Not a point estimate. A full distribution.

In [ ]:
posterior_mu = trace.posterior['mu_growth'].values.flatten()

print("=== Posterior: True Revenue Growth Rate ===")
print(f"  Posterior mean:         {posterior_mu.mean():.2%}")
print(f"  Posterior median:       {np.median(posterior_mu):.2%}")
print(f"  Posterior std:          {posterior_mu.std():.2%}")
print(f"  90% credible interval:  [{np.percentile(posterior_mu, 5):.2%}, {np.percentile(posterior_mu, 95):.2%}]")
print()
print("  --- Probability Statements ---")
print(f"  P(growth > 8%):  {(posterior_mu > 0.08).mean():.1%}")
print(f"  P(growth > 5%):  {(posterior_mu > 0.05).mean():.1%}")
print(f"  P(growth < 3%):  {(posterior_mu < 0.03).mean():.1%}")

# Plot: Prior vs Posterior
fig, ax = plt.subplots()
ax.hist(prior_samples, bins=80, alpha=0.4, color='#6B8CCC', label='Prior (sector-level)', density=True)
ax.hist(posterior_mu, bins=80, alpha=0.8, color='#1F3A7A', label='Posterior (after data)', density=True)
ax.axvline(posterior_mu.mean(), color='#C8960C', linewidth=2.5,
           label=f'Posterior mean: {posterior_mu.mean():.2%}')
ax.axvline(np.percentile(posterior_mu, 5), color='gray', linewidth=1.5, linestyle='--')
ax.axvline(np.percentile(posterior_mu, 95), color='gray', linewidth=1.5, linestyle='--',
           label=f'90% CI: [{np.percentile(posterior_mu, 5):.1%}, {np.percentile(posterior_mu, 95):.1%}]')
ax.set_title('Prior vs. Posterior: True Revenue Growth Rate $\\theta$', fontweight='bold')
ax.set_xlabel('Revenue Growth Rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 6 — Posterior Predictive: Next Quarter Forecast (Section 1.3)

The posterior predictive distribution $p(\tilde{y} \mid y)$ gives the full probability distribution over what next quarter's revenue growth could be — averaging over all plausible values of $\theta$, weighted by the posterior.

This is the direct input to your DCF terminal value distribution.

In [ ]:
ppc_samples = ppc.posterior_predictive['observed'].values.flatten()

print("=== Posterior Predictive: Next Quarter Revenue Growth ===")
print(f"  Forecast median:        {np.median(ppc_samples):.2%}")
print(f"  90% forecast interval:  [{np.percentile(ppc_samples, 5):.2%}, {np.percentile(ppc_samples, 95):.2%}]")
print(f"  P(next quarter > 8%):   {(ppc_samples > 0.08).mean():.1%}")
print(f"  P(next quarter < 5%):   {(ppc_samples < 0.05).mean():.1%}")

fig, ax = plt.subplots()
ax.hist(ppc_samples, bins=80, color='#2E7D32', alpha=0.8, edgecolor='white', linewidth=0.2)
ax.axvline(np.median(ppc_samples), color='#C8960C', linewidth=2.5,
           label=f'Median forecast: {np.median(ppc_samples):.2%}')
ax.axvline(np.percentile(ppc_samples, 5), color='gray', linewidth=1.5, linestyle='--')
ax.axvline(np.percentile(ppc_samples, 95), color='gray', linewidth=1.5, linestyle='--',
           label=f'90% interval')
ax.set_title('Posterior Predictive $p(\\tilde{y} \\mid y)$: Next Quarter Revenue Growth Forecast', fontweight='bold')
ax.set_xlabel('Forecasted Growth Rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.show()

---

## Summary

What just happened, end to end:

1. **Prior** — we encoded sector-level structural knowledge as a Normal distribution over the true growth rate $\theta$. Before seeing this company's data, we believed growth was probably around 7%.

2. **Likelihood** — we specified that each observed quarter is a draw from a Normal distribution centered on the true growth rate, with some noise.

3. **Posterior** — PyMC ran NUTS and returned thousands of samples from the posterior distribution over $\theta$. The prior got updated by 8 quarters of earnings data into a sharper, more company-specific estimate.

4. **Posterior predictive** — we used the posterior to generate a distribution over next quarter's revenue growth — a full probability distribution, not a point estimate.

This is the Chapter 1 loop. Chapter 2 introduces conjugate models that compute the posterior analytically — and the Beta-Binomial model for default and win-rate estimation.